# Ngày 22 · Perceptron và MLP

- Cài quy tắc cập nhật perceptron và thấy nó hội tụ khi dữ liệu tách được tuyến tính
- Thấy perceptron thất bại trên XOR
- Viết lan truyền xuôi của MLP hai tầng với ReLU và softmax ổn định số

**Đọc:**
- Rosenblatt (1958), The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain, Psychological Review 65(6), 386–408
- ISLR (bản 2), §10.1 Single Layer Neural Networks và §10.2 Multilayer Neural Networks
- Goodfellow, Bengio & Courville (2016), Deep Learning, Chương 6 Deep Feedforward Networks (§6.1 Example: Learning XOR)

In [ ]:
import json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
from scipy.special import softmax as _sp_softmax
from sklearn.datasets import load_iris
from sklearn.linear_model import Perceptron
_iris = load_iris()
X_ir = _iris.data[:, 2:4]; y_ir = np.where(_iris.target == 0, 1, -1)       # setosa: tách được tuyến tính
X_xor = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]]); y_xor = np.array([-1, 1, 1, -1])
_sk22 = Perceptron(shuffle=False, eta0=1.0, alpha=0, penalty=None, max_iter=100, tol=None).fit(X_ir, y_ir)
_r22 = np.random.default_rng(22)
_p22 = {'W1': _r22.normal(size=(4, 5)), 'b1': _r22.normal(size=5), 'W2': _r22.normal(size=(5, 3)), 'b2': _r22.normal(size=3)}
_X22 = _r22.normal(size=(7, 4))


## Quy tắc học của perceptron

Gợi ý: Duyệt mẫu theo thứ tự. Nếu y·(w·x + b) ≤ 0 thì w ← w + y·x, b ← b + y. Đếm số lần cập nhật trong mỗi vòng; vòng nào bằng 0 thì dừng.

In [ ]:
def perceptron_train(X, y, epochs=100):
    # y ∈ {−1, +1}. Trả (w, b, mistakes) với mistakes là list số lần cập nhật ở từng vòng.
    raise NotImplementedError

def perceptron_predict(w, b, X):
    # Trả mảng ±1 (score = 0 tính là +1).
    raise NotImplementedError


In [ ]:
_report('d22-perceptron',
    _check('Ví dụ tính tay: w = [1, 3], b = 0 sau 2 vòng', lambda: ((lambda r: np.allclose(r[0], [1, 3]) and r[1] == 0 and list(r[2]) == [2, 0])(perceptron_train(np.array([[2., 1.], [-1., -1.], [1., -2.]]), np.array([1, -1, -1]))), 'Cập nhật khi y·(w·x + b) ≤ 0.')),
    _check('Iris setosa: hội tụ, khớp sklearn Perceptron', lambda: ((lambda r: r[2][-1] == 0 and np.allclose(r[0], _sk22.coef_[0]) and np.isclose(r[1], _sk22.intercept_[0]) and np.all(perceptron_predict(r[0], r[1], X_ir) == y_ir))(perceptron_train(X_ir, y_ir)), 'shuffle=False, eta0=1.')),
    _check('XOR: không bao giờ hết lỗi', lambda: (min(perceptron_train(X_xor, y_xor, 50)[2]) > 0, 'Không có đường thẳng nào tách được XOR.')),
)


## Lan truyền xuôi của MLP hai tầng

Gợi ý: softmax: trừ max theo hàng trước khi exp. Lan truyền xuôi: Z1 = X @ W1 + b1, H = ReLU(Z1), Z2 = H @ W2 + b2, P = softmax(Z2). Cross-entropy: −mean(log P[i, y_i]) có chặn p ≥ eps.

In [ ]:
def relu(z):
    raise NotImplementedError

def softmax(Z):
    # Theo từng hàng, không tràn số khi Z rất lớn.
    raise NotImplementedError

def mlp_forward(X, params):
    # params: W1 (d, h), b1 (h,), W2 (h, C), b2 (C,). Trả (P, cache) với cache = {'X', 'Z1', 'H'}.
    raise NotImplementedError

def cross_entropy(P, y, eps=1e-12):
    raise NotImplementedError


In [ ]:
_xor_params = {'W1': np.array([[1., 1.], [1., 1.]]), 'b1': np.array([0., -1.]), 'W2': np.array([[0., 10.], [0., -20.]]), 'b2': np.array([0., -5.])}
_report('d22-mlp',
    _check('softmax ổn định và mỗi hàng cộng bằng 1', lambda: ((lambda P: np.all(np.isfinite(P)) and np.allclose(P, _sp_softmax(np.array([[1000., 1001., 999.], [1., 2., 3.]]), axis=1)))(softmax(np.array([[1000., 1001., 999.], [1., 2., 3.]]))), 'Z = 1000 không được sinh NaN.')),
    _check('Lan truyền xuôi khớp cách tính tham chiếu', lambda: ((lambda P: np.allclose(P, _sp_softmax(np.maximum(0, _X22 @ _p22['W1'] + _p22['b1']) @ _p22['W2'] + _p22['b2'], axis=1)))(mlp_forward(_X22, _p22)[0]), '4 → 5 → 3, ReLU.')),
    _check('Với trọng số cho sẵn, MLP giải được XOR', lambda: (np.array_equal(mlp_forward(X_xor, _xor_params)[0].argmax(1), [0, 1, 1, 0]), 'h₁ = ReLU(x₁ + x₂), h₂ = ReLU(x₁ + x₂ − 1).')),
    _check('Cross-entropy đúng và hữu hạn khi p = 0', lambda: (np.isclose(cross_entropy(np.array([[.7, .3], [.2, .8]]), np.array([0, 1])), -np.mean(np.log([.7, .8]))) and np.isfinite(cross_entropy(np.array([[1., 0.]]), np.array([1]))), 'Chặn xác suất trước khi lấy log.')),
)


## Dự án

Mở notebook cloud Ngay_25_26_28_CNN (Kaggle hoặc Colab) và chạy phần tải Fashion-MNIST. Huấn luyện LogisticRegression trên ảnh làm phẳng để có mốc so sánh cho cả tuần; ghi accuracy và ma trận nhầm lẫn, đánh dấu hai lớp hay bị nhầm nhất.